In [1]:
# %pip install numpy pandas scanpy matplotlib statsmodels scikit-misc
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import misc
import statsmodels.api as sm
import statsmodels.formula.api as smf

C:\Users\miel_\AppData\Local\Temp\ipykernel_10064\799035524.py:7: DeprecationWarning: scipy.misc is deprecated and will be removed in 2.0.0
  from scipy import misc


In [3]:
# Configuración de Scanpy para graficación limpia
sc.settings.verbosity = 3             # Nivel de detalle de logs
sc.settings.set_figure_params(dpi=80, facecolor='white')

C:\Users\miel_\AppData\Local\Temp\ipykernel_10064\2100244685.py:3: FutureWarning: The method set_figure_params is deprecated and will be removed in the future. Use :func:`scanpy.set_figure_params` instead
  sc.settings.set_figure_params(dpi=80, facecolor='white')


In [4]:
# ==============================================================================
# MÓDULO 1: OBTENCIÓN Y DESCARGA DE DATOS DE SINGLE-CELL RNA-SEQ (30 min)
# ==============================================================================
print("=== MÓDULO 1: OBTENCIÓN Y DESCARGA DE DATOS DE SINGLE-CELL RNA-SEQ ===")

# Descargamos el dataset público 'pbmc3k' de NCBI GEO / 10x Genomics directamente desde Scanpy
# Contiene ~2,700 células mononucleares de sangre periférica
adata = sc.datasets.pbmc3k()
print(f"Dataset cargado exitosamente. Matriz original: {adata.n_obs} células x {adata.n_vars} genes\n")

=== MÓDULO 1: OBTENCIÓN Y DESCARGA DE DATOS DE SINGLE-CELL RNA-SEQ ===
Dataset cargado exitosamente. Matriz original: 2700 células x 32738 genes



In [5]:
# ==============================================================================
# MÓDULO 2: CONTROL DE CALIDAD Y FILTRADO DE CÉLULAS/GENES (45 min)
# ==============================================================================
print("=== MÓDULO 2: CONTROL DE CALIDAD Y FILTRADO DE CÉLULAS/GENES ===")

# 1. Identificar genes mitocondriales (indicadores de estrés o lisis celular)
adata.var['mt'] = adata.var_names.str.startswith('MT-')  

# 2. Calcular métricas globales de Control de Calidad (QC)
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

# 3. Visualización de métricas de calidad iniciales
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True, show=False)
plt.savefig("1_qc_violins.png", bbox_inches='tight')
plt.close()

# 4. Filtrado de células de baja calidad o dobletes, y genes no expresados
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata = adata[adata.obs.pct_counts_mt < 5, :].copy() # Retener células con < 5% de ADNmt

print(f"Matriz tras filtrado de QC: {adata.n_obs} células x {adata.n_vars} genes\n")

=== MÓDULO 2: CONTROL DE CALIDAD Y FILTRADO DE CÉLULAS/GENES ===
filtered out 19024 genes that are detected in less than 3 cells
Matriz tras filtrado de QC: 2643 células x 13714 genes



In [6]:
print("=== MÓDULO 3: NORMALIZACIÓN, REDUCCIÓN DE DIMENSIONALIDAD Y CLUSTERING ===")

# 1. Normalización a 10,000 lecturas y transformación logarítmica
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# 2. Guardar la matriz normalizada en adata.raw
adata.raw = adata

# 3. Selección de genes altamente variables con 'seurat'
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5, flavor='seurat')

sc.pl.highly_variable_genes(adata, show=False)
plt.savefig("2_highly_variable_genes.png", bbox_inches='tight')
plt.close()

# 4. Filtrar la matriz para conservar solo los HVGs
adata = adata[:, adata.var.highly_variable].copy()

# 5. Escalamiento de datos y PCA
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver='arpack')

# 6. Grafo de vecinos, UMAP y Clustering de Leiden
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=20)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.5)

# Visualizar y guardar mapa UMAP
sc.pl.umap(adata, color=['leiden'], show=False)
plt.savefig("3_umap_clusters.png", bbox_inches='tight')
plt.close()

print("Reducción de dimensionalidad UMAP y clustering de Leiden completados.")

=== MÓDULO 3: NORMALIZACIÓN, REDUCCIÓN DE DIMENSIONALIDAD Y CLUSTERING ===
normalizing counts per cell
    finished (0:00:02)
extracting highly variable genes
    finished (0:00:01)
--> added
    'highly_variable', boolean vector (adata.var)
    'means', float vector (adata.var)
    'dispersions', float vector (adata.var)
    'dispersions_norm', float vector (adata.var)
computing PCA
    with n_comps=50


c:\Users\miel_\AppData\Local\Programs\Python\Python312\Lib\functools.py:909: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


    finished (0:00:00)
computing neighbors
    using 'X_pca' with n_pcs = 20


c:\Users\miel_\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:09)
computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (0:00:02)
running Leiden clustering
    finished: found 9 clusters and added
    'leiden', the cluster labels (adata.obs, categorical) (0:00:00)


C:\Users\miel_\AppData\Local\Temp\ipykernel_10064\3084707177.py:27: FutureWarning: The `igraph` implementation of leiden clustering is *orders of magnitude faster*. Set the flavor argument to (and install if needed) 'igraph' to use it.
In the future, the default backend for leiden will be igraph instead of leidenalg. To achieve the future defaults please pass: `flavor='igraph'` and `n_iterations=2`. `directed` must also be `False` to work with igraph’s implementation.
  sc.tl.leiden(adata, resolution=0.5)


Reducción de dimensionalidad UMAP y clustering de Leiden completados.


In [ ]:
# ==============================================================================
# MÓDULO 4: EXPRESIÓN DIFERENCIAL (SINGLE-CELL Y BULK RNA-SEQ) (45 min)
# ==============================================================================
print("\n=== MÓDULO 4: EXPRESIÓN DIFERENCIAL (SINGLE-CELL Y BULK RNA-SEQ) ===")

# 1. Marcadores por Clúster con Scanpy (Prueba no paramétrica de Wilcoxon)
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')
sc.pl.rank_genes_groups(adata, n_genes=5, sharey=False, show=False)
plt.savefig("4_marker_genes_per_cluster.png", bbox_inches='tight')
plt.close()

# 2. Expresión Diferencial Bulk-style mediante Regresión Binomial Negativa (Statsmodels)
# Compatible nativamente con Python 3.12+ sin dependencias de C/C++ obsoletas
np.random.seed(42)
genes = [f'GEN_{i+1}' for i in range(20)]
muestras = [f'Muestra_{i+1}' for i in range(6)]
counts_sim = np.random.negative_binomial(n=10, p=0.3, size=(20, 6))

df_bulk = pd.DataFrame(counts_sim.T, index=muestras, columns=genes)
df_bulk['condition'] = ['Control', 'Control', 'Control', 'Tratamiento', 'Tratamiento', 'Tratamiento']

# Ajuste de Modelo Lineal Generalizado (GLM) Binomial Negativo gen por gen
de_results = []
for gen in genes:
    formula = f"{gen} ~ C(condition, Treatment(reference='Control'))"
    model = smf.glm(formula=formula, data=df_bulk, family=sm.families.NegativeBinomial()).fit()
    
    p_val = model.pvalues.iloc[1]
    log2FC = model.params.iloc[1]
    de_results.append({'Gene': gen, 'log2FC': log2FC, 'pvalue': p_val})

df_de_results = pd.DataFrame(de_results).sort_values('pvalue')
print("\nPrimeros resultados de Expresión Diferencial Bulk (GLM Binomial Negativo en Statsmodels):")
print(df_de_results.head())

print("\n=== SCRIPT COMPLETADO CON ÉXITO. GRÁFICAS GUARDADAS EN EL DIRECTORIO LOCAL. ===")


=== MÓDULO 4: EXPRESIÓN DIFERENCIAL (SINGLE-CELL Y BULK RNA-SEQ) ===
ranking genes
    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:07)

Primeros resultados de Expresión Diferencial Bulk (GLM Binomial Negativo en Statsmodels):
      Gene    log2FC    pvalue
12  GEN_13  0.441833  0.596672
10  GEN_11 -0.411735  0.622147
8    GEN_9 -0.336472  0.685327
1    GEN_2  0.311436  0.709712
4    GEN_5 -0.266268  0.749707

=== SCRIPT COMPLETADO CON ÉXITO. GRÁFICAS GUARDADAS EN EL DIRECTORIO LOCAL. ===


C:\Users\miel_\AppData\Local\Temp\ipykernel_10064\875837702.py:26: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  model = smf.glm(formula=formula, data=df_bulk, family=sm.families.NegativeBinomial()).fit()
C:\Users\miel_\AppData\Local\Temp\ipykernel_10064\875837702.py:26: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  model = smf.glm(formula=formula, data=df_bulk, family=sm.families.NegativeBinomial()).fit()
C:\Users\miel_\AppData\Local\Temp\ipykernel_10064\875837702.py:26: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  model = smf.glm(formula=formula, data=df_bulk, family=sm.families.NegativeBinomial()).fit()
C:\Users\miel_\AppData\Local\Temp\ipykernel_10064\875837702.py:26: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  model = smf.glm(formula=formula, data=df_bulk, famil